# Lección 8: El Algoritmo Simplex
### Álgebra Lineal Computacional y Fundamentos de la Física Matemática

---

Este cuaderno de Jupyter Colab aborda de manera formal y computacional el **Algoritmo Simplex**, el método de optimización lineal por excelencia para resolver problemas con múltiples variables y restricciones. Desarrollado por George Dantzig en 1947, este algoritmo revolucionó la matemática aplicada y la investigación de operaciones. A lo largo de esta lección, exploraremos la fundamentación teórica, su interpretación geométrica en politopos de varias dimensiones, las reglas de pivotaje algebraicas, y los métodos de inicialización. Concluiremos con la implementación de un simulador del simplex paso a paso en Python, una visualización geométrica interactiva en 3D para el problema de producción y la verificación utilizando la biblioteca científica `scipy.optimize.linprog`.

---


## Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Comprender** el funcionamiento del algoritmo Simplex y su transición entre soluciones factibles básicas.
2. **Interpretar** geométricamente el comportamiento del algoritmo a través de vértices y aristas de un politopo.
3. **Aplicar** de forma rigurosa las reglas de pivotaje (selección de variable entrante y saliente mediante análisis de ratios).
4. **Distinguir** los métodos de inicialización (Método de la Gran M y Método Bifásico/Dos Fases) para problemas con restricciones complejas.
5. **Implementar** un código en Python que simule la transición de tablas simplex paso a paso.
6. **Visualizar** la región factible y la trayectoria del Simplex en problemas de 2 y 3 variables utilizando gráficos interactivos 2D y 3D.
7. **Verificar** los resultados obtenidos computacionalmente con la biblioteca científica `scipy`.


## 1. Fundamentos del Algoritmo Simplex

Un problema general de programación lineal en su **forma estándar** (o canónica de minimización) se escribe como:

$$\min \quad c^T x$$
$$\text{s.a.} \quad A x = b$$
$$x \ge 0$$

Donde $A \in \mathbb{R}^{m \times n}$ es la matriz de coeficientes tecnológicos (típicamente con $m < n$ y rango completo de filas $\text{rang}(A) = m$), $b \in \mathbb{R}^m$ es el vector de requerimientos del lado derecho ($b \ge 0$), y $c \in \mathbb{R}^n$ es el vector de coeficientes de la función objetivo.

### 1.1 Definiciones Básicas

* **Conjunto Factible:** El conjunto $F = \{x \in \mathbb{R}^n \mid A x = b, \ x \ge 0\}$ es un conjunto convexo.
* **Solución Factible Básica (SFB):** Se obtiene al seleccionar $m$ columnas linealmente independientes de la matriz $A$ para formar una base $B \in \mathbb{R}^{m \times m}$. Si definimos las variables correspondientes a estas columnas como **variables básicas** ($x_B$) y las $n - m$ restantes como **variables no básicas** ($x_N = 0$), la solución se halla resolviendo el sistema:
  $$B x_B = b \implies x_B = B^{-1} b$$
  Si además $x_B \ge 0$, entonces la solución es factible y representa un **vértice (o punto esquina)** del conjunto factible $F$.
* **Arista o Borde:** Una trayectoria lineal que une dos vértices adyacentes del conjunto factible. Algebraicamente, representa un cambio en la base donde una variable no básica entra y una variable básica sale.
* **Punto Esquina:** La intersección de $n$ hiperplanos que delimitan la región factible. En $\mathbb{R}^n$, un punto esquina es la representación geométrica de una SFB.


## 2. Reglas de Pivotaje e Iteración

El algoritmo Simplex es un método iterativo que inicia en una SFB (un vértice) y se mueve a lo largo de las aristas hacia vértices adyacentes que mejoren el valor de la función objetivo. Este proceso se rige por tres reglas principales:

### 2.1 Regla de entrada en la base (Entering Variable Rule)
El método utiliza los **costos reducidos** (o coeficientes de la fila Z) para determinar si la solución actual es óptima y, en caso contrario, qué variable no básica $x_k$ debe entrar en la base para mejorar el objetivo:
* **Maximización:** Se selecciona la variable $x_k$ con el coeficiente más negativo en la fila Z (o costo reducido positivo más grande, dependiendo de la notación de la tabla). El criterio formal es:
  $$k = \text{ArgMax} \{ c_j - z_j \mid c_j - z_j > 0 \}$$
* **Minimización:** Se introduce la variable con el costo reducido negativo de mayor magnitud:
  $$k = \text{ArgMax} \{ c_j - z_j \mid c_j - z_j < 0 \}$$

### 2.2 Regla de salida de la base (Leaving Variable Rule)
Cuando una variable $x_k$ entra a la base, su valor aumenta desde cero. Para mantener la factibilidad de las variables básicas ($x_{Bi} \ge 0$), se realiza la **prueba de la razón mínima (análisis de ratios)**.
Si expresamos las variables básicas en función de la entrante:
$$x_B = B^{-1}b - B^{-1}a_k x_k = x_B^0 - y_k x_k$$

El crecimiento de $x_k$ está bloqueado por la primera variable básica que llega a valer cero. Esto se calcula dividiendo la columna del lado derecho ($b$ o $x_B^0$) entre los elementos estrictamente positivos de la columna de la variable entrante ($y_{ik}$):
$$x_k = \min_{i \in V_B, y_{ik} > 0} \left\{ \frac{x_{Bi}^0}{y_{ik}} \right\}$$

La fila $r$ que alcanza este mínimo define la variable básica saliente $x_{Br}$, la cual pasará a ser no básica (con valor cero) en la siguiente iteración.
* **Caso No Acotado:** Si para la variable entrante $x_k$ se cumple que todos los $y_{ik} \le 0$ para todo $i$, entonces $x_k$ puede crecer indefinidamente sin violar la factibilidad. En este caso, el problema es **no acotado** y la función objetivo tiende a infinito.

### 2.3 Prueba de Optimización (Optimality Test)
* En problemas de **maximización**, la solución es óptima si todos los costos reducidos (fila objetivo) son menores o iguales a cero ($c_j - z_j \le 0$).
* En problemas de **minimización**, es óptima si todos son mayores o iguales a cero ($c_j - z_j \ge 0$).


## 3. Inicialización del Algoritmo

Para comenzar el algoritmo Simplex, se requiere una Solución Factible Básica inicial. Si todas las restricciones del problema original son de tipo menor o igual ($\le$) con términos independientes no negativos ($b \ge 0$), las **variables de holgura** proporcionan inmediatamente una base identidad factible inicial ($x_B = b$, con $x_N = 0$).

Sin embargo, si existen restricciones de tipo mayor o igual ($\ge$) o de igualdad ($=$), el origen no es factible. En estos casos, se recurre a variables artificiales y a métodos de penalización:

### 3.1 Método de la Gran M (Big-M Method)
Consiste en introducir variables artificiales $a_i \ge 0$ en las restricciones que lo requieran para formar una base identidad inicial. Para garantizar que estas variables salgan de la base en la solución final, se les asigna una gran penalización $M > 0$ en la función objetivo:
* **Maximización:** Se resta $M \cdot a_i$ de la función objetivo.
* **Minimización:** Se suma $M \cdot a_i$ a la función objetivo.

Durante las iteraciones del Simplex, el gran valor de $M$ obliga al algoritmo a reducir las variables artificiales a cero.

### 3.2 Método de las Dos Fases (Two-Phase Method)
Divide la resolución en dos etapas:
* **Fase I:** Se define una función objetivo artificial que consiste en minimizar la suma de todas las variables artificiales:
  $$\min \quad W = \sum a_i$$
  Se resuelve este problema auxiliar usando el Simplex. Si al finalizar la Fase I el valor mínimo de $W$ es cero ($W^* = 0$), significa que existe una solución factible para el problema original. Se eliminan las columnas de las variables artificiales y se procede a la Fase II. Si $W^* > 0$, el problema original es **no factible** (incompatible).
* **Fase II:** Utilizando la base factible obtenida en la Fase I, se restaura la función objetivo original y se continúa con el Simplex tradicional hasta hallar el óptimo.


## 4. Caso Práctico 1: Resolución Gráfica y Simplex Paso a Paso

Consideramos el siguiente problema de maximización:

$$\text{Maximizar } Z = 3x + 2y$$
$$\text{sujeto a:}$$
$$2x + y \le 18 \quad \text{(Restricción 1)}$$
$$2x + 3y \le 42 \quad \text{(Restricción 2)}$$
$$3x + y \le 24 \quad \text{(Restricción 3)}$$
$$x, y \ge 0$$

### 4.1 Evaluación de Vértices mediante Método Gráfico
Al graficar y resolver el sistema, obtenemos los siguientes puntos esquina de la región factible:
* **$O(0, 0)$** $\implies Z = 0$
* **$C(0, 14)$** $\implies Z = 28$
* **$G(3, 12)$** $\implies Z = 33$ *(Óptimo)*
* **$H(6, 6)$** $\implies Z = 30$
* **$F(8, 0)$** $\implies Z = 24$

### 4.2 Aclaración Académica de la Errata del PDF
En la **Tabla III (Iteración 3)** en la página 8 del PDF oficial, se observa un error de cálculo/tipográfico en el coeficiente de la variable de holgura $P_5$ en la fila de la variable básica $P_4$. El texto muestra un coeficiente de **$-7$** en lugar de **$4$**.
Veamos la deducción algebraica correcta:
En la iteración 2, la fila correspondiente a $P_4$ (Fila 2) tiene los coeficientes:
$$R_2 = [0, 7/3, 0, 1, -2/3 \mid 26]$$
La fila pivote elegida es la Fila 1 ($P_3$, correspondiente a la variable entrante $P_2$), cuyos coeficientes antes de escalar son $[0, 1/3, 1, 0, -2/3 \mid 2]$. Tras dividir entre el elemento pivote $1/3$, la nueva Fila 1 escalada es:
$$R_1' = [0, 1, 3, 0, -2 \mid 6]$$
Para eliminar el coeficiente de $P_2$ en la Fila 2, realizamos la operación de fila Gauss-Jordan:
$$R_2 \leftarrow R_2 - \frac{7}{3} R_1'$$
Calculando para la columna de $P_5$:
$$P_{5,\text{nuevo}} = -\frac{2}{3} - \frac{7}{3}(-2) = -\frac{2}{3} + \frac{14}{3} = \frac{12}{3} = 4$$
Por lo tanto, el valor correcto es **$4$**, no $-7$. 
A continuación, graficaremos la región factible y luego programaremos el simulador Simplex en Python para verificar de forma exacta toda la secuencia de tablas.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definición de restricciones para graficar límites
# 2x + y = 18   => y = 18 - 2x
# 2x + 3y = 42  => y = (42 - 2x)/3
# 3x + y = 24   => y = 24 - 3x

x_vals = np.linspace(0, 10, 400)
y_r1 = 18 - 2 * x_vals
y_r2 = (42 - 2 * x_vals) / 3.0
y_r3 = 24 - 3 * x_vals

plt.figure(figsize=(10, 8))

# Graficación de las líneas
plt.plot(x_vals, y_r1, 'r-', label='R1: $2x + y \\le 18$')
plt.plot(x_vals, y_r2, 'g-', label='R2: $2x + 3y \\le 42$')
plt.plot(x_vals, y_r3, 'b-', label='R3: $3x + y \\le 24$')

# Sombras de la región factible
y_factible = np.minimum(y_r1, y_r2)
y_factible = np.minimum(y_factible, y_r3)
y_factible = np.maximum(y_factible, 0)
plt.fill_between(x_vals, 0, y_factible, where=(x_vals <= 8), color='purple', alpha=0.25, label='Región Factible (Acotada)')

# Vértices del polígono
vertices_2d = [(0, 0), (8, 0), (6, 6), (3, 12), (0, 14)]
beneficios_2d = [3*v[0] + 2*v[1] for v in vertices_2d]

for v, ben in zip(vertices_2d, beneficios_2d):
    color = 'gold' if v == (3, 12) else 'black'
    plt.scatter(v[0], v[1], color=color, s=120, zorder=5)
    plt.annotate(f"Vértice {v}\nZ = {ben}", xy=v, xytext=(v[0]+0.3, v[1]+0.3),
                 fontsize=9, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8, ec="gray"))

# Dibujar la trayectoria del Simplex
# Trayectoria: (0,0) -> (8,0) -> (6,6) -> (3,12)
simplex_path_x = [0, 8, 6, 3]
simplex_path_y = [0, 0, 6, 12]
plt.plot(simplex_path_x, simplex_path_y, 'k--', linewidth=2.5, label='Trayectoria Simplex')
plt.quiver(simplex_path_x[:-1], simplex_path_y[:-1], 
           np.diff(simplex_path_x), np.diff(simplex_path_y), 
           scale_units='xy', angles='xy', scale=1, color='orange', width=0.007, zorder=4)

plt.title('Método Gráfico y Trayectoria del Simplex (Ejemplo 1)', fontsize=12, fontweight='bold')
plt.xlabel('Variable x', fontsize=10)
plt.ylabel('Variable y', fontsize=10)
plt.xlim(-1, 10)
plt.ylim(-1, 16)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right')
plt.show()



## 5. Caso Práctico 2: Optimización en la Producción de Cocinetas

Una empresa fabrica tres modelos de cocinetas: $M1$ ($x_1$), $M2$ ($x_2$) y $M3$ ($x_3$).
* **Beneficio por unidad:** M1: $5000, M2: $4000, M3: $3000 (expresado en miles: $5, 4, 3$).
* **Restricción de Almacenamiento:** No se pueden almacenar más de 28 unidades en total.
  $$x_1 + x_2 + x_3 \le 28$$
* **Restricción de Tiempo de Fabricación:** Fabricar M1 requiere 3 semanas, M2 requiere 2 semanas y M3 requiere 1 semana. El tiempo disponible total es de 48 semanas.
  $$3x_1 + 2x_2 + x_3 \le 48$$
* **No negatividad:** $x_1, x_2, x_3 \ge 0$

Modelado Matemático:
$$\text{Maximizar } P = 5x_1 + 4x_2 + 3x_3$$
$$\text{sujeto a:}$$
$$x_1 + x_2 + x_3 \le 28 \quad \text{(Almacenamiento)}$$
$$3x_1 + 2x_2 + x_3 \le 48 \quad \text{(Semanas de producción)}$$

A continuación implementamos el resolutor Simplex completo paso a paso en Python. El programa mostrará de forma limpia las tablas asociadas a cada iteración.


In [ ]:
import numpy as np

class SimplexSolver:
    def __init__(self, c, A, b, var_names, slack_names, maximization=True):
        self.c = np.array(c, dtype=float)
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        self.max = maximization
        self.m, self.n = self.A.shape
        
        # Nombres de variables para visualización
        self.var_names = list(var_names) + list(slack_names)
        
        # Construcción del Tableau inicial:
        # Contiene coeficientes de variables originales + variables de holgura + columna b
        self.tableau = np.zeros((self.m + 1, self.n + self.m + 1))
        self.tableau[:self.m, :self.n] = self.A
        self.tableau[:self.m, self.n:self.n+self.m] = np.eye(self.m)
        self.tableau[:self.m, -1] = self.b
        
        # Fila Z
        if self.max:
            self.tableau[-1, :self.n] = -self.c
        else:
            self.tableau[-1, :self.n] = self.c
            
        # Base inicial (índices de las variables de holgura)
        self.basis = [self.n + i for i in range(self.m)]
        self.iter_count = 0
        
    def print_tableau(self):
        cols = ["Base"] + self.var_names + ["b"]
        header = " | ".join(f"{col:>8}" for col in cols)
        divider = "-" * len(header)
        
        print(divider)
        print(f"Tabla Simplex - Iteración {self.iter_count}")
        print(divider)
        print(header)
        print(divider)
        
        for i in range(self.m):
            base_var = self.var_names[self.basis[i]]
            row_str = f"{base_var:>8}"
            for val in self.tableau[i]:
                row_str += f" | {val:>8.3f}"
            print(row_str)
            
        print(divider)
        z_str = f"{'Z':>8}"
        for val in self.tableau[-1]:
            z_str += f" | {val:>8.3f}"
        print(z_str)
        print(divider)
        
    def step(self):
        # Fila Z
        z_row = self.tableau[-1, :-1]
        
        # Comprobación de optimalidad
        if self.max:
            if np.all(z_row >= -1e-9):
                print("\n¡Solución óptima encontrada!")
                return False
        else:
            if np.all(z_row <= 1e-9):
                print("\n¡Solución óptima encontrada!")
                return False
                
        # Selección de variable entrante (Regla de Bland / Valor absoluto más negativo)
        entering_col = np.argmin(z_row)
        
        # Prueba de la razón mínima para variable saliente
        col_vals = self.tableau[:-1, entering_col]
        rhs_vals = self.tableau[:-1, -1]
        
        ratios = []
        for i in range(self.m):
            if col_vals[i] > 1e-9:
                ratios.append(rhs_vals[i] / col_vals[i])
            else:
                ratios.append(np.inf)
                
        if np.all(np.isinf(ratios)):
            print("\nEl problema no está acotado.")
            return False
            
        leaving_row = np.argmin(ratios)
        pivot = self.tableau[leaving_row, entering_col]
        
        print(f"-> Variable entrante: {self.var_names[entering_col]}")
        print(f"-> Variable saliente: {self.var_names[self.basis[leaving_row]]}")
        print(f"-> Elemento pivote: {pivot:.4f}")
        
        # Operaciones de fila Gauss-Jordan
        self.tableau[leaving_row, :] /= pivot
        for r in range(self.m + 1):
            if r != leaving_row:
                factor = self.tableau[r, entering_col]
                self.tableau[r, :] -= factor * self.tableau[leaving_row, :]
                
        # Actualización de la base
        self.basis[leaving_row] = entering_col
        self.iter_count += 1
        return True

    def solve(self):
        self.print_tableau()
        while self.step():
            self.print_tableau()
        self.print_solution()
        
    def print_solution(self):
        print("\nValores de las variables en el óptimo:")
        sol = {name: 0.0 for name in self.var_names}
        for i, idx in enumerate(self.basis):
            sol[self.var_names[idx]] = self.tableau[i, -1]
        for name in self.var_names[:self.n]:
            print(f"  {name} = {sol[name]:.4f}")
        print(f"Valor óptimo de la función objetivo: Z = {self.tableau[-1, -1]:.4f}")

# 1. RESOLVER EJEMPLO 1 (Verificación de Errata en Tabla III)
print("================ EJEMPLO 1 PASO A PASO ================")
c1 = [3.0, 2.0]
A1 = [[2.0, 1.0], [2.0, 3.0], [3.0, 1.0]]
b1 = [18.0, 42.0, 24.0]
solver1 = SimplexSolver(c1, A1, b1, ["x", "y"], ["y1", "y2", "y3"])
solver1.solve()

# 2. RESOLVER PROBLEMA DE LAS COCINETAS
print("\n================ EJEMPLO 2 (COCINETAS) ================")
c2 = [5.0, 4.0, 3.0]
A2 = [[1.0, 1.0, 1.0], [3.0, 2.0, 1.0]]
b2 = [28.0, 48.0]
solver2 = SimplexSolver(c2, A2, b2, ["x1", "x2", "x3"], ["y1", "y2"])
solver2.solve()



### 5.1 Visualización del Poliedro Factible de 3 Variables en 3D

Para el problema de las cocinetas, las variables son $x_1, x_2, x_3$. Geométricamente, la región factible define un poliedro convexo en el espacio tridimensional $\mathbb{R}^3$.
Los planos de restricción que limitan el poliedro son:
* Almacenamiento: $x_1 + x_2 + x_3 = 28$
* Producción: $3x_1 + 2x_2 + x_3 = 48$
* Planos de no negatividad: $x_1 = 0$, $x_2 = 0$, $x_3 = 0$.

Graficaremos las superficies límite y marcaremos de color dorado la trayectoria que realiza el Simplex:
$$O(0,0,0) \longrightarrow F(16,0,0) \longrightarrow G(10,0,18)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Crear grilla en x1 y x2
x1_grid = np.linspace(0, 20, 100)
x2_grid = np.linspace(0, 25, 100)
X1, X2 = np.meshgrid(x1_grid, x2_grid)

# Plano 1 (Almacenamiento): x3 = 28 - x1 - x2
Z_almacen = 28 - X1 - X2
# Plano 2 (Producción): x3 = 48 - 3*x1 - 2*x2
Z_produccion = 48 - 3*X1 - 2*X2

# Filtrar para representar solo la región factible (x3 >= 0 y donde cada plano es el restrictivo)
Z_almacen_fact = np.where((Z_almacen <= Z_produccion) & (Z_almacen >= 0), Z_almacen, np.nan)
Z_produccion_fact = np.where((Z_produccion <= Z_almacen) & (Z_produccion >= 0), Z_produccion, np.nan)

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Graficar caras factibles de las restricciones
ax.plot_surface(X1, X2, Z_almacen_fact, alpha=0.4, color='cyan', label='Almacenamiento')
ax.plot_surface(X1, X2, Z_produccion_fact, alpha=0.4, color='orange', label='Producción')

# Vértices del poliedro en 3D
vertices_3d = np.array([
    [0, 0, 0],    # O
    [16, 0, 0],   # F
    [10, 0, 18],  # G (Óptimo)
    [0, 0, 28],   # C
    [0, 20, 8],   # D
    [0, 24, 0]    # E
])

# Graficar los vértices
ax.scatter(vertices_3d[:,0], vertices_3d[:,1], vertices_3d[:,2], color='black', s=100, depthshade=False, zorder=5)

# Etiquetas de los vértices
labels = ['O(0,0,0)', 'F(16,0,0)', 'G(10,0,18) Óptimo', 'C(0,0,28)', 'D(0,20,8)', 'E(0,24,0)']
alignments = [('left', 'top'), ('left', 'bottom'), ('right', 'bottom'), ('left', 'bottom'), ('left', 'bottom'), ('left', 'bottom')]
for i in range(len(vertices_3d)):
    ha, va = alignments[i]
    ax.text(vertices_3d[i,0], vertices_3d[i,1], vertices_3d[i,2], labels[i], 
            fontweight='bold', fontsize=9, ha=ha, va=va)

# Graficar la trayectoria del Simplex
# O(0,0,0) -> F(16,0,0) -> G(10,0,18)
trayectoria = np.array([
    [0, 0, 0],
    [16, 0, 0],
    [10, 0, 18]
])
ax.plot(trayectoria[:,0], trayectoria[:,1], trayectoria[:,2], color='gold', linewidth=4, label='Trayectoria Simplex', zorder=10)

ax.set_title('Poliedro Factible de 3D y Trayectoria Simplex (Cocinetas)', fontsize=13, fontweight='bold')
ax.set_xlabel('M1 (x1)', fontsize=11)
ax.set_ylabel('M2 (x2)', fontsize=11)
ax.set_zlabel('M3 (x3)', fontsize=11)

ax.set_xlim(-1, 20)
ax.set_ylim(-1, 28)
ax.set_zlim(-1, 30)

# Rotar la vista para una mejor apreciación
ax.view_init(elev=20, azim=45)

plt.show()



## 6. Verificación Científica con SciPy

Para comprobar la robustez y exactitud matemática de las implementaciones personalizadas, utilizaremos la biblioteca `scipy.optimize.linprog` para resolver directamente el problema de las cocinetas.


In [ ]:
from scipy.optimize import linprog

# coeficientes de la función objetivo: Z = 5x1 + 4x2 + 3x3
# Como SciPy minimiza por defecto, multiplicamos los coeficientes por -1
c_scipy = [-5.0, -4.0, -3.0]

# Matriz A de restricciones de desigualdad (<=)
A_scipy = [
    [1.0, 1.0, 1.0],  # x1 + x2 + x3 <= 28
    [3.0, 2.0, 1.0]   # 3x1 + 2x2 + x3 <= 48
]

# Vector b
b_scipy = [28.0, 48.0]

# Límites de no negatividad
limites = [(0, None), (0, None), (0, None)]

# Ejecutar solver
res = linprog(c_scipy, A_ub=A_scipy, b_ub=b_scipy, bounds=limites, method='highs')

print("--- RESULTADOS OBTENIDOS POR SCIPY ---")
print(f"Estado de la resolución: {res.message}")
print(f"Púnto óptimo hallado:")
print(f"  x1 (M1) = {res.x[0]:.4f}")
print(f"  x2 (M2) = {res.x[1]:.4f}")
print(f"  x3 (M3) = {res.x[2]:.4f}")
print(f"Valor óptimo de la ganancia: Z = {-res.fun:.4f}")



## 7. Resumen de la Lección

1. **Soluciones Factibles Básicas (SFB):** Representan algebraicamente los puntos esquina o vértices de la región factible y se construyen eligiendo $m$ variables básicas linealmente independientes.
2. **Reglas de Pivote:** El Simplex avanza iterativamente seleccionando una variable no básica entrante (costo reducido óptimo) y una variable básica saliente (obtenida mediante la prueba de la razón mínima para no violar la factibilidad).
3. **Inicialización:** Problemas con restricciones de tipo $\ge$ o $=$ requieren el uso de variables artificiales iniciales, las cuales se eliminan mediante el método de la **Gran M** (penalización) o el método de las **Dos Fases** (minimización de la suma de artificiales en la Fase I).
4. **Visualización en 3D:** Para problemas con 3 variables de decisión, la región factible define un poliedro en el espacio tridimensional $\mathbb{R}^3$, donde el algoritmo Simplex realiza un recorrido físico sobre las aristas exteriores de dicho poliedro.
5. **Detección de Erratas:** Mediante álgebra lineal rigurosa, se verificó y clarificó que la Tabla III del PDF posee una errata tipográfica mostrando un coeficiente de $-7$ en lugar de $4$ en la columna de la variable $P_5$, confirmando la necesidad de validación matemática computacional en ciencia.

---

## 8. Referencias Bibliográficas

1. Sotelo, J. C. (2012). *Álgebra lineal para estudiantes de ingeniería y ciencias*. Ed. 1, México: Interamericana Editores, SA de CV.
2. O’Connor, J. J. (1997). *Técnicas de cálculo para sistemas de ecuaciones, programación lineal y programación entera*. Madrid: Ed. Reverte, Escuela Técnica Superior de Ingenieros Industriales.
3. Blázquez, A. (2020). *El método simplex. Algoritmo de las dos fases*. Universidad de Valencia.
4. Guía Digital PHPSimplex. (2020). *Teoría del método gráfico*. http://www.phpsimplex.com/teoria_metodo_grafico.htm
5. Wikipedia, La enciclopedia libre. *Algoritmo de pivote / Método Simplex*.
6. Yapura, M. (2002). *El método simplex para resolver problemas de programación lineal: una síntesis*.
7. Universidad Nacional de La Plata (UNLP). (2017). *El método Simplex*. http://www.mate.unlp.edu.ar/practicas/66_13_0804200912835.pdf
